> 📝 **Planning note (author use -- remove before publishing)**
>
> **Intended contents:** auto-metadata, compliance checks (IOOS + custom), adding custom metadata three ways, dtype/fill-value finalisation, export.
>
> **To do:**
> - ~~Point at the edited file~~, ~~add the export step~~ (both done).
> - Write the section intros -- the notebook runs end to end but has almost no narrative.
> - **Note for readers:** `compliance_checks_ioos` needs the optional `compliance-checker` package, which is not installed on Windows. Say so, or Windows readers hit an ImportError with no explanation.
> - `set_glob_attr` / `set_var_attr` prompt for input, so "Restart & Run All" blocks there. Worth warning about.
> - Check the final compliance report is clean (or explain what remains).

<img src="../../data/images/hiaoos_learning_moored.png" width="300" align="right">


# Preparing data for publication

The state of the art for publishing ocean data is the NetCDF format with community standard metadata conventions (CF and ACDD).

*Notebook 10 of 10 in the series*  
[← 9. Combining instruments](./09_combining_and_correlating_instruments.ipynb)

In [1]:
from kval.data import moored
from kval.metadata import conventionalize, io

In [2]:
ds = moored.load_nc('../../data/moored_CTD_test_data/intermediate_data/AT200_21_22_SBE37_20773_49m_edited.nc')

___

## Add some metadata

- What `metadata_auto` fills in for you, and what it cannot know.

> ✏️ Placeholder -- prose needed.

In [3]:
ds = moored.metadata_auto(ds)

In [4]:
ds

<xarray.Dataset> Size: 1MB
Dimensions:    (TIME: 31729)
Coordinates:
  * TIME       (TIME) float64 254kB 1.894e+04 1.894e+04 ... 1.927e+04 1.927e+04
    LATITUDE   float32 4B 81.41
    LONGITUDE  float32 4B 31.24
Data variables:
    PRES       (TIME) float64 254kB ...
    CNDC       (TIME) float64 254kB ...
    TEMP       (TIME) float64 254kB ...
    PSAL       (TIME) float64 254kB ...
Attributes: (12/36)
    history:                         2021-11-07 - 2022-10-06: Data collection...
    date_created:                    2026-09-25T01:59:37Z
    source:                          Subsurface mooring
    instrument:                      In Situ/Laboratory Instruments>Profilers...
    instrument_model:                Sea-Bird SBE37SMP-RS232
    instrument_serial_number:        20773
    ...                              ...
    keywords_vocabulary:             GCMD Science Keywords 9.1.5
    platform_vocabulary:             GCMD Platform Keywords Version 9.1.5
    iso_topic_category:              oceans
    Conventions:                     ACDD-1.3, CF-1.11
    SBE_processing_date:             2022-10-06T19:09:06Z
    SBE_flags_applied:               yes

___

## Run quick checks

- What CF and ACDD are, and why compliance matters for published data.
- The difference between the IOOS checker (external, standards-based) and kval's custom check (NPI practice).
- **Warning for Windows readers:** the IOOS checker is an optional dependency that is not installed on Windows -- the cell below will raise an ImportError. The custom check works everywhere.

> ✏️ Placeholder -- prose needed. The Windows caveat is not optional.

In [5]:
moored.compliance_checks_ioos(ds)

In [6]:
moored.compliance_checks_custom(ds)


----------------------------------
⚠️ Dataset has 5 issue(s)
----------------------------------

❌❌❌ MISSING REQUIRED GLOBAL ATTRIBUTES ❌❌❌
title, summary, creator_name, creator_email, institution
⮕ These MUST be added for CF/ACDD compliance

⚠️ 64-bit types (32-bit is recommended):
PRES, CNDC, TEMP, PSAL, TIME
   ⮕ Suggestion: use kval.conventionalize.convert_64_to_32(ds)

⚠️ Suspicious/missing _FillValue (including NaNs, which are discouraged):
PRES, CNDC, TEMP, PSAL
   ⮕ Suggestion: use kval.conventionalize.nans_to_fill_value(ds)

❌ Missing 'processing_level':
PRES, CNDC, TEMP, PSAL
   ⮕ Suggestion: add globally or on all relevant variables

⚠️ Missing 'QC_indicator' (not strictly required):
PRES, CNDC, TEMP, PSAL

⚠️ Missing recommended global attributes:
product_version, id, license, project, doi, acknowledgment, references, data_set_progress, cruise_name, area, location, data_assembly_center, creator_type, creator_url, creator_institution, publisher_name, publisher_url, publishe

___

## Add some custom metadata

- Three ways to do this, shown below: directly in xarray, via helper functions, and from a YAML file.
- Which one you would actually use in practice, and why (the file is reproducible; the prompts are not).

> ✏️ Placeholder -- prose needed.

#### In xarray

There are a few ways to add attributes. You can set them directly in `xarray`:

In [7]:
ds.attrs['title'] = 'Mock title' # Set a global attribute
ds.CNDC.attrs['long_name'] = 'Sea water electrical conductivity' # Set a variable-level attribute

#### Using kval helper functions

... or use some helper functions if you prefer. These prompt you for the value:

In [8]:
ds = conventionalize.set_glob_attr(ds, 'title')

In [9]:
ds = conventionalize.set_var_attr(ds, 'TEMP', 'standard_name')

#### Using input file

Perhaps the easiest way to do this is to import metadata from an editable textfile (`.yaml`). Support functions in `lkval.metadata.io` allow importing metadata directly into your dataset. 

> **Note** The `.yaml` file has to follow the prescribed format exactly - small formatting errors will make this crash.

Here, we'll first imort a file containing global attributes;

In [10]:
ds = io.import_metadata(ds, 'metadata_input/custom_global_metadata.yaml')

.. and then from a second file containg variable attributes. Note that you can just as well combine the two in a single `.yaml` file.

In [11]:
ds = io.import_metadata(ds, 'metadata_input/custom_variable_metadata.yaml')

In [12]:
ds

<xarray.Dataset> Size: 1MB
Dimensions:    (TIME: 31729)
Coordinates:
  * TIME       (TIME) float64 254kB 1.894e+04 1.894e+04 ... 1.927e+04 1.927e+04
    LATITUDE   float32 4B 81.41
    LONGITUDE  float32 4B 31.24
Data variables:
    PRES       (TIME) float64 254kB ...
    CNDC       (TIME) float64 254kB ...
    TEMP       (TIME) float64 254kB ...
    PSAL       (TIME) float64 254kB ...
Attributes: (12/66)
    history:                         2021-11-07 - 2022-10-06: Data collection...
    date_created:                    2026-09-25T01:59:37Z
    source:                          Subsurface mooring
    instrument:                      In Situ/Laboratory Instruments>Profilers...
    instrument_model:                Sea-Bird SBE37SMP-RS232
    instrument_serial_number:        20773
    ...                              ...
    publisher_url:                   TBW
    publisher_email:                 TBW
    publisher_type:                  TBW
    publisher_institution:           TBW
    naming_authority:                TBW
    license:                         CC-BY 4.0

### Finalize dataset

A couple of steps

In [13]:
ds = moored.add_now_as_date_created(ds)

Convert 64 bit integers and floats to 32 bit (safe in typical oceanographic applications) 

In [14]:
ds = moored.convert_64_to_32(ds)

Instead of NaNs, use a fillvalue (default `-9999.0`). NaNs are acceptable, but can in some cases cause problems. The `FillValue` is clearly defined and part of CF/ADCC, and is the safest approach.  

In [15]:
ds = moored.nans_to_fill_value(ds)

In [16]:
moored.compliance_checks_ioos(ds)

In [17]:
moored.compliance_checks_custom(ds)


----------------------------------
⚠️ Dataset has 1 issue(s)
----------------------------------

❌ Attributes with TBW placeholders:
   – Global:
    area, location, date_issued, project, acknowledgment, comment, references, institution, data_assembly_center, creator_name, creator_type, creator_email, creator_institution, creator_url, publisher_name, publisher_url, publisher_email, publisher_type, publisher_institution, naming_authority
   ⮕ Suggestion: Replace with actual metadata

----------------------------------
✅ Passed checks
----------------------------------

✅ No 64-bit variable types

✅ _FillValue nicely defined for all relevant variables

✅ Required 'processing_level' present correctly

✅ Recommended 'QC_indicator' present correctly

✅ All relevant variables have 'units' and 'standard_name' or 'long_name' attributes

✅ No 'SBE_FLAG' variable present

✅ Dimension coordinates are finite and monotonic

✅ All coordinates (except STATION) have coverage_content_type='coordinate'

### Export the final file

> ✏️ **Review note** — the notebook stopped at the compliance check and never wrote anything out, which seemed like the one thing a "preparing data for publication" notebook has to end with. Added the export below -- adjust the path and filename to whatever you want the published product to be called.

In [19]:
out_path = '../../data/moored_CTD_test_data/publishable_data'
out_name = 'AT200_21_22_temp_sal_pres_15252_49m_v1.nc'

moored.to_netcdf(ds, out_path, out_name)

Updated history attribute. Current content:
---
2021-11-07 - 2022-10-06: Data collection.
2022-10-06: Processed to .cnv using SBE software.
2026-09-25: Post-processing.
2026-09-25: Creation of this netcdf file.
---
Exported NetCDF file as: ../../data/moored_CTD_test_data/publishable_data/AT200_21_22_temp_sal_pres_15252_49m_v1.nc


___

**End of the series.** [↩ Back to the introduction](./01_introduction.ipynb)